In [9]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Coffee Shop Orders")      # run once; next time use gc.open("Coffee Shop Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "pickup_time", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1VtJW3Zo9onuAWnT1Wlc5Bt-V0ghCz6SCKOYdivVTLgE


In [10]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"espresso": 120, "cappuccino": 180, "latte": 200, "cold coffee": 220}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}

class Order(BaseModel):
    """A coffee order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["espresso", "cappuccino", "latte", "cold coffee"] = Field(description="Coffee drink from the menu")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date or time as the customer said it")


In [11]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one coffee drink.

    Args:
        item: Coffee drink, e.g. "espresso", "cappuccino", "latte" or "cold coffee"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the coffee shop's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str) -> str:
    """Save a confirmed coffee order to the Coffee Shop Orders sheet.
    Use only when the customer has given their name, drink, quantity and pickup date or time."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date, total])
    return f"Saved: {order.qty} x {order.item} for {order.name} at {order.date}, Rs {total}."

In [12]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for a coffee shop.
FACTS:  Menu drinks: espresso, cappuccino, latte and cold coffee.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, drink, quantity and pickup date or time; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on friday? How much are 4 cappuccinos?"))
print(chat("I'm Siddu. Please order 2 cold coffee for 6 Dec at 10 am."))
print(chat("What coffees do you have on the menu?"))

Yes, we’re open on Friday from 9 am to 8 pm. Four cappuccinos will cost you Rs 720. Let me know your name and pickup time, and I’ll log the order for you!
✅ Order confirmed!  
2 cold coffee for Siddu on 6 Dec at 10 am – total Rs 440.  
See you then!
We offer four tasty options:  
- Espresso  
- Cappuccino  
- Latte  
- Cold coffee  

Let me know which one you’d like!
